# 📊 Análise de Churn – Telecom X

Este projeto tem como objetivo analisar os fatores que influenciam o churn (cancelamento) de clientes e identificar padrões que auxiliem na retenção.

🔹 Célula 1 – Importar e ler os dados

In [30]:
import pandas as pd

url = "https://raw.githubusercontent.com/alura-cursos/challenge2-data-science/refs/heads/main/TelecomX_Data.json"
df = pd.read_json(url)

## 🔎 Exploração Inicial dos Dados

🔹 Célula 2 – Explorar o dado bruto

In [31]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerID  7267 non-null   object
 1   Churn       7267 non-null   object
 2   customer    7267 non-null   object
 3   phone       7267 non-null   object
 4   internet    7267 non-null   object
 5   account     7267 non-null   object
dtypes: object(6)
memory usage: 340.8+ KB


## 🧹 Tratamento e Normalização dos Dados (ETL)

🔹 Célula 3 – Normalizar o JSON

In [32]:
df_normalizado = pd.json_normalize(df.to_dict(orient="records"))

🔹 Célula 4 – Ajustar nomes das colunas

In [33]:
df_normalizado.columns = df_normalizado.columns.str.replace('.', '_')

In [34]:
df = df_normalizado.copy()

🔹 Célula 5 – Converter Charges Total (a que você achou que faltava)

In [35]:
df['account_Charges_Total'] = pd.to_numeric(
    df['account_Charges_Total'],
    errors='coerce'

)

🔹 Célula 6 – Verificar valores nulos

In [36]:
df.isnull().sum()

customerID                    0
Churn                         0
customer_gender               0
customer_SeniorCitizen        0
customer_Partner              0
customer_Dependents           0
customer_tenure               0
phone_PhoneService            0
phone_MultipleLines           0
internet_InternetService      0
internet_OnlineSecurity       0
internet_OnlineBackup         0
internet_DeviceProtection     0
internet_TechSupport          0
internet_StreamingTV          0
internet_StreamingMovies      0
account_Contract              0
account_PaperlessBilling      0
account_PaymentMethod         0
account_Charges_Monthly       0
account_Charges_Total        11
dtype: int64

🔹 Célula 7 – Converter Churn para numérico

In [37]:
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## 📊 Análise Exploratória (EDA)

## Análise exploratória

Nesta etapa, foram realizadas análises iniciais para entender o comportamento
do churn em relação a algumas variáveis dos clientes.

In [38]:
df['Churn'].value_counts(normalize=True) * 100

Churn
0.0    73.463013
1.0    26.536987
Name: proportion, dtype: float64

In [39]:
df.columns

Index(['customerID', 'Churn', 'customer_gender', 'customer_SeniorCitizen',
       'customer_Partner', 'customer_Dependents', 'customer_tenure',
       'phone_PhoneService', 'phone_MultipleLines', 'internet_InternetService',
       'internet_OnlineSecurity', 'internet_OnlineBackup',
       'internet_DeviceProtection', 'internet_TechSupport',
       'internet_StreamingTV', 'internet_StreamingMovies', 'account_Contract',
       'account_PaperlessBilling', 'account_PaymentMethod',
       'account_Charges_Monthly', 'account_Charges_Total'],
      dtype='object')

### 📄 Churn por Tipo de Contrato

In [40]:
tabela_contrato = pd.crosstab(df["account_Contract"], df["Churn"], normalize="index") * 100
tabela_contrato.round(2)

Churn,0.0,1.0
account_Contract,,
Month-to-month,57.29,42.71
One year,88.73,11.27
Two year,97.17,2.83


In [41]:
df["account_Contract"].value_counts(normalize=True) * 100

account_Contract
Month-to-month    55.112151
Two year          23.985138
One year          20.902711
Name: proportion, dtype: float64

In [42]:
### 💳 Churn por Método de Pagamento

In [43]:
tabela_pagamento = pd.crosstab(df["account_PaymentMethod"], df["Churn"], normalize="index") * 100
tabela_pagamento.round(2)

Churn,0.0,1.0
account_PaymentMethod,,
Bank transfer (automatic),83.29,16.71
Credit card (automatic),84.76,15.24
Electronic check,54.71,45.29
Mailed check,80.89,19.11


### 🌐 Churn por Tipo de Internet

In [44]:
tabela_internet = pd.crosstab(df["internet_InternetService"], df["Churn"], normalize="index") * 100
tabela_internet.round(2)

Churn,0.0,1.0
internet_InternetService,,
DSL,81.04,18.96
Fiber optic,58.11,41.89
No,92.60,7.40


In [45]:
### ⏳ Churn por Tempo de Permanência

In [46]:
df["customer_tenure"].describe()

count    7267.000000
mean       32.346498
std        24.571773
min         0.000000
25%         9.000000
50%        29.000000
75%        55.000000
max        72.000000
Name: customer_tenure, dtype: float64

In [47]:
df.groupby("Churn")["customer_tenure"].mean()

Churn
0.0    37.569965
1.0    17.979133
Name: customer_tenure, dtype: float64

In [48]:
df["tenure_group"] = pd.cut(
    df["customer_tenure"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-12 meses", "12-24 meses", "24-48 meses", "48-72 meses"]
)

In [49]:
pd.crosstab(df["tenure_group"], df["Churn"], normalize="index") * 100

Churn,0.0,1.0
tenure_group,,
0-12 meses,52.321839,47.678161
12-24 meses,71.289062,28.710938
24-48 meses,79.611041,20.388959
48-72 meses,90.486824,9.513176


In [50]:
### 💰 Churn por Valor Mensal

In [51]:
df.groupby("Churn")["account_Charges_Monthly"].mean()

Churn
0.0    61.265124
1.0    74.441332
Name: account_Charges_Monthly, dtype: float64

In [52]:
df.groupby("Churn")["account_Charges_Monthly"].describe()

,count,mean,std,min,25%,50%,75%,max
Churn,,,,,,,,
0.0,5174.0,61.265124,31.092648,18.25,25.10,64.425,88.4,118.75
1.0,1869.0,74.441332,24.666053,18.85,56.15,79.650,94.2,118.35


📌 Conclusão da Análise

A análise exploratória revelou que o churn está fortemente relacionado ao tipo de contrato, método de pagamento, tempo de permanência e valor mensal.

Clientes com contrato mensal apresentaram taxa de cancelamento superior a 41%, enquanto contratos de dois anos reduziram o churn para menos de 3%.

O método de pagamento via electronic check apresentou a maior taxa de evasão (43%), indicando possível menor fidelização.

Observou-se ainda que clientes com internet fibra óptica e mensalidades mais elevadas demonstram maior propensão ao cancelamento.

Além disso, o período mais crítico ocorre nos primeiros 12 meses, onde o churn atinge 46%, evidenciando a necessidade de estratégias de retenção no início do relacionamento com o cliente.